# Build your own controller

This notebook teaches **Option A**: a Python controller drives the production
`EngineSession` (Rust kernel) in a simple loop:

1. Read belief + pipeline from `session.snapshot()`
2. Choose an order with your `controller.order(ctx)`
3. Advance physics + filter with `session.step(order_qty)`

We compare a **naive base-stock** baseline and a small **tabular Q-learning**
starter against production **damped survival-weighted** ordering via
`session.act(policy="damped_sw")`.

Fixtures use `smoke_cool_shipments()` only (no Abdella parquet). This path does **not**
use `sim.episode.run_closed_loop_episode`.

## Setup

From the repo root with the Rust extension built:

```bash
uv sync --all-extras --python 3.11
uv run --python 3.11 maturin develop --release -m crates/voi_py/Cargo.toml
export BLUEBERRIES_VOI_BACKEND=rust
uv run jupyter lab notebooks/
```

Import the library helpers and confirm the Rust backend is available.

In [ ]:
%matplotlib inline

from __future__ import annotations

import os
from collections import defaultdict
from statistics import mean

os.environ.setdefault("BLUEBERRIES_VOI_BACKEND", "rust")

import matplotlib.pyplot as plt

from blueberries_voi.backend import rust_available, warn_fallback_once
from blueberries_voi.controller import (
    ControllerStepLog,
    EpisodeTotals,
    NaiveBaseStockController,
    TabularQLearningController,
    default_session_config,
    episode_totals_from_logs,
    run_act_episode,
    run_controller_episode,
    run_controller_session,
)
from blueberries_voi.simulator import EngineSession

plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.3})

warn_fallback_once()
if not rust_available():
    raise RuntimeError("Build blueberries_voi._core and set BLUEBERRIES_VOI_BACKEND=rust")
print("Rust backend ready.")

### Simulation parameters

These knobs match the Studio demo budgets. Smaller particle counts run faster in a
notebook while keeping the same calendar and filter structure. Production damped-SW
uses `n_rollout_paths=0` (no nested rollout scoring).

In [ ]:
# Primary seed for single-episode cumulative charts
SEED = 42

# Seeds for paired multi-policy evaluation (identical SESSION_CFG + seed per arm)
EVAL_SEEDS = [11, 23, 37, 41, 53]

# Damped survival-weighted policy knobs for act(policy="damped_sw")
SW_ALPHA = 0.9
SW_RHO = 0.8

# Scored horizon (calendar days per controller run)
N_DAYS = 28

# Training episodes for tabular Q-learning
TRAIN_EPISODES = 8

# Particle filter size (lower = faster notebook)
N_PARTICLES = 64

# Observation scenario preset (P1 = production default)
OBS_SCENARIO = "P1"

SESSION_CFG = default_session_config(
    n_particles=N_PARTICLES,
    n_rollout_paths=0,
    obs_scenario=OBS_SCENARIO,
)

### Controller parameters

The naive controller tops up to a fixed shelf target. The Q-learning starter explores
discrete order quantities on a coarse (weekday, on-hand bin) grid.

In [ ]:
# Naive base-stock target (units on shelf + pipeline)
BASE_STOCK_TARGET = 48

# Case size for naive rounding (matches ModelParams default)
CASE_SIZE = 8

# Discrete order actions for Q-learning (units, not cases)
QL_ACTIONS = [0, 8, 16, 24, 32]

# ε-greedy exploration probability during training
QL_EPSILON = 0.2

# Q-learning step size
QL_LEARNING_RATE = 0.15

# Discount factor (0 = maximize immediate day profit)
QL_DISCOUNT = 0.0

# On-hand cap for binning tabular states
QL_MAX_ON_HAND = 80

# Number of on-hand bins for tabular states
QL_ON_HAND_BINS = 8

## Train tabular Q-learning

Each training episode resets the session, runs `run_controller_session`, and calls
`observe` with **day profit** as the reward signal.

In [ ]:
ql = TabularQLearningController(
    QL_ACTIONS,
    epsilon=QL_EPSILON,
    learning_rate=QL_LEARNING_RATE,
    discount=QL_DISCOUNT,
    max_on_hand=QL_MAX_ON_HAND,
    on_hand_bins=QL_ON_HAND_BINS,
    seed=SEED,
)

train_profits: list[float] = []
for ep in range(TRAIN_EPISODES):
    session = EngineSession()
    session.init(SESSION_CFG, seed=SEED + ep)
    logs = run_controller_session(session, ql, N_DAYS)
    ep_profit = sum(log.day_profit for log in logs)
    train_profits.append(ep_profit)
    print(f"train episode {ep + 1}/{TRAIN_EPISODES}: profit={ep_profit:.1f}")

ql.epsilon = 0.0  # greedy evaluation after training

## Multi-seed benchmark vs damped SW

After training, score **greedy Q-learning**, **naive base-stock**, and Rust
**damped SW** on the same evaluation seeds. Each seed fixes the demand path so
profit, spoilage, and stockout comparisons are **paired** (common random numbers).

In [ ]:
naive = NaiveBaseStockController(target_units=BASE_STOCK_TARGET, case_size=CASE_SIZE)

ql_results: list[EpisodeTotals] = []
naive_results: list[EpisodeTotals] = []
sw_results: list[EpisodeTotals] = []

for seed in EVAL_SEEDS:
    ql_results.append(
        run_controller_episode(SESSION_CFG, ql, seed, N_DAYS, policy_label="Q-learning")
    )
    naive_results.append(
        run_controller_episode(SESSION_CFG, naive, seed, N_DAYS, policy_label="Naive")
    )
    sw_results.append(
        run_act_episode(
            SESSION_CFG,
            seed,
            N_DAYS,
            policy="damped_sw",
            alpha=SW_ALPHA,
            rho=SW_RHO,
            policy_label="Damped SW",
        )
    )
    print(
        f"seed={seed}: Q={ql_results[-1].profit:.1f} "
        f"naive={naive_results[-1].profit:.1f} SW={sw_results[-1].profit:.1f}"
    )

### Fig 1 — paired slopegraph (Q-learning vs damped SW)

Each gray segment connects the **same seed** under greedy Q-learning (left) and
damped SW (right). Upward slopes on profit mean Q-learning beat SW on that path.

## Cumulative profit (single seed)

One scored episode at `SEED` for day-by-day cumulative profit curves.

In [ ]:
def daily_profits_controller(controller, seed: int) -> list[float]:
    session = EngineSession()
    session.init(SESSION_CFG, seed=seed)
    logs = run_controller_session(session, controller, N_DAYS)
    return [log.day_profit for log in logs]


def daily_profits_damped_sw(seed: int) -> list[float]:
    session = EngineSession()
    session.init(SESSION_CFG, seed=seed)
    profits: list[float] = []
    for _ in range(N_DAYS):
        delta = session.act(policy="damped_sw")
        log = ControllerStepLog.from_delta(delta)
        profits.append(log.day_profit)
    return profits


naive_daily = daily_profits_controller(naive, SEED)
ql_daily = daily_profits_controller(ql, SEED)
damped_daily = daily_profits_damped_sw(SEED)

print(f"Naive total profit: {sum(naive_daily):.1f}")
print(f"Q-learning total profit: {sum(ql_daily):.1f}")
print(f"Damped SW total profit: {sum(damped_daily):.1f}")

### Cumulative profit comparison

In [ ]:
def cumulative(series: list[float]) -> list[float]:
    out: list[float] = []
    total = 0.0
    for x in series:
        total += x
        out.append(total)
    return out


fig, ax = plt.subplots()
days = list(range(1, len(naive_daily) + 1))
ax.plot(days, cumulative(naive_daily), label="Naive base-stock", color="#4C72B0")
ax.plot(days, cumulative(ql_daily), label="Tabular Q-learning", color="#55A868")
ax.plot(days, cumulative(damped_daily), label="Damped SW (act)", color="#C44E52", ls="--")
ax.set_xlabel("Day")
ax.set_ylabel("Cumulative day profit")
ax.set_title(f"Controller comparison (seed={SEED}, smoke cool shipments)")
ax.legend()
fig.tight_layout()
plt.show()

### Mean profit across paired seeds

In [ ]:
by_policy: dict[str, list[float]] = defaultdict(list)
for row in benchmark_rows:
    by_policy[row.policy_label].append(row.profit)

labels = ["naive", "q_learning", "damped_sw"]
means = [mean(by_policy[label]) for label in labels]
display_labels = ["Naive base-stock", "Tabular Q-learning", "Damped SW (act)"]
colors = ["#4C72B0", "#55A868", "#C44E52"]

fig, ax = plt.subplots()
ax.bar(display_labels, means, color=colors)
ax.set_ylabel("Mean episode profit")
ax.set_title(f"Paired-seed benchmark ({len(BENCHMARK_SEEDS)} seeds × {N_DAYS} days)")
fig.tight_layout()
plt.show()

## Next steps

- Swap `TabularQLearningController` for your own subclass of `ControllerTemplate`.
- Feed richer state from `ControllerContext.belief` (f-marginals per lot).
- Tune `BASE_STOCK_TARGET` or Q-learning actions against the same `run_controller_session` loop.
- Read ADR 0148 for why this path differs from `sim.episode.run_closed_loop_episode`.